# Tugas 1

In [1]:
%pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm

In [3]:
# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)

kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },
    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}
print("Konfigurasi berhasil")
print("Sport :", kategori["sport"]["url"])
print("Finance:", kategori["finance"]["url"])

Konfigurasi berhasil
Sport : https://sport.detik.com/indeks
Finance: https://finance.detik.com/indeks


In [4]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """
    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

In [5]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):
    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):
        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:
            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:
            print("Error:", e)
            continue

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)

            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:
                    sudah_ada.add(href)
                    links.append(href)

        print(
            "Jumlah link:",
            len(links)
        )

        if len(links) >= target:
            break

        # delay
        time.sleep(
            random.uniform(1, 2)
        )

    return links

In [6]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 39
Membaca halaman 3
Jumlah link: 57
Membaca halaman 4
Jumlah link: 77
Membaca halaman 5
Jumlah link: 97
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 157
Total kandidat Sport: 157


In [7]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 17
Membaca halaman 2
Jumlah link: 35
Membaca halaman 3
Jumlah link: 55
Membaca halaman 4
Jumlah link: 74
Membaca halaman 5
Jumlah link: 91
Membaca halaman 6
Jumlah link: 110
Membaca halaman 7
Jumlah link: 130
Membaca halaman 8
Jumlah link: 147
Membaca halaman 9
Jumlah link: 162
Total kandidat Finance: 162


In [8]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 17
Membaca halaman 2
Jumlah link: 35
Membaca halaman 3
Jumlah link: 55
Membaca halaman 4
Jumlah link: 74
Membaca halaman 5
Jumlah link: 91
Membaca halaman 6
Jumlah link: 110
Membaca halaman 7
Jumlah link: 130
Membaca halaman 8
Jumlah link: 147
Membaca halaman 9
Jumlah link: 162
Total kandidat Finance: 162


In [9]:
def scraping_kategori(
    links,
    label,
    target=100
):
    hasil = []

    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):

        if len(hasil) >= target:
            break

        isi = ambil_isi_berita(url)

        if isi is not None:
            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )

        # Delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )

        print(
            f"{label} berhasil:",
            len(hasil)
        )

    return hasil

In [10]:
def bersihkan_teks(teks):
    if teks is None:
        return None

    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )

    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )

    teks = re.sub(
        r"\s+",
        " ",
        teks
    )

    return teks.strip()

In [11]:
def ambil_isi_berita(url):
    try:
        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()

        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )

        isi = bersihkan_teks(isi)

        if isi and len(isi) >= 300:
            return isi

    except Exception as e:
        print("Gagal:", url)
        print("Error:", e)

    return None

In [12]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)

Scraping sport:   1%|▍                                                              | 1/157 [00:01<03:47,  1.46s/it]

sport berhasil: 0


Scraping sport:   1%|▊                                                              | 2/157 [00:03<04:36,  1.79s/it]

sport berhasil: 1


Scraping sport:   2%|█▏                                                             | 3/157 [00:05<04:47,  1.87s/it]

...

sport berhasil: 98


Scraping sport:  66%|████████████████████████████████████████                     | 103/157 [03:05<01:37,  1.80s/it]

sport berhasil: 99


Scraping sport:  66%|████████████████████████████████████████▍                    | 104/157 [03:07<01:35,  1.81s/it]

sport berhasil: 100


In [13]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)

Scraping finance:   1%|▍                                                            | 1/162 [00:02<05:32,  2.06s/it]

finance berhasil: 1


Scraping finance:   1%|▊                                                            | 2/162 [00:04<05:27,  2.05s/it]

finance berhasil: 2


Scraping finance:   2%|█▏                                                           | 3/162 [00:06<05:27,  2.06s/it]

...

finance berhasil: 98


Scraping finance:  63%|█████████████████████████████████████▏                     | 102/162 [03:01<01:33,  1.56s/it]

finance berhasil: 99


Scraping finance:  64%|█████████████████████████████████████▌                     | 103/162 [03:03<01:45,  1.78s/it]

finance berhasil: 100


In [14]:
data_semua = data_sport + data_finance

df = pd.DataFrame(data_semua)

In [15]:
print("Total data:", len(df))
print("\nJumlah data berdasarkan label:")
print(df["label"].value_counts())

Total data: 200

Jumlah data berdasarkan label:
label
sport      100
finance    100
Name: count, dtype: int64


In [16]:
df.head(10)

,isi_berita,label
0,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport
1,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,Di tengah viral insiden cepirit pada ajang Hyr...,sport
3,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,sport
4,Bagas Maulana/Apriyani Rahayu tak minder meski...,sport
5,Ajang lari bergengsi Alfamart Run 2026 siap di...,sport
6,Pasangan baru ganda campuran Dejan Ferdinansya...,sport
7,Grid MotoGP 2027 akhirnya lengkap. Senna Agius...,sport
8,Ganda putri Febriana Dwipuji Kusuma/Meilysa Tr...,sport
9,MotoGP 2026 berlanjut ke Austria akhir pekan i...,sport


In [17]:
df.shape

(200, 2)

In [18]:
df.columns

Index(['isi_berita', 'label'], dtype='str')

In [19]:
df.insert(
    0,
    "id",
    range(1, len(df) + 1)
)

In [20]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [21]:
df.head()

,id,isi_berita,label
0,1,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,3,Di tengah viral insiden cepirit pada ajang Hyr...,sport
3,4,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,sport
4,5,Bagas Maulana/Apriyani Rahayu tak minder meski...,sport


In [22]:
print("Total data:", len(df))
print(df["label"].value_counts())

Total data: 200
label
sport      100
finance    100
Name: count, dtype: int64


In [23]:
df.head(10)

,id,isi_berita,label
0,1,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport
1,2,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
2,3,Di tengah viral insiden cepirit pada ajang Hyr...,sport
3,4,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,sport
4,5,Bagas Maulana/Apriyani Rahayu tak minder meski...,sport
5,6,Ajang lari bergengsi Alfamart Run 2026 siap di...,sport
6,7,Pasangan baru ganda campuran Dejan Ferdinansya...,sport
7,8,Grid MotoGP 2027 akhirnya lengkap. Senna Agius...,sport
8,9,Ganda putri Febriana Dwipuji Kusuma/Meilysa Tr...,sport
9,10,MotoGP 2026 berlanjut ke Austria akhir pekan i...,sport


In [24]:
nama_file = "dataset_detik_200_berita.xlsx"

df.to_excel(
    nama_file,
    index=False
)

print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx
